# 3. Compare saved runs
Reads completed run artifacts without training. Runs must reference identical
saved benchmark files. Validation is the default comparison split. Freeze model
choices before enabling test reporting; do not select seeds or features by test results.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from hepml.adapters.research import (
    read_settings, inspect_inputs, feature_preview, save_provenance,
    benchmark_directory, prepare_benchmark, train_benchmark,
    stage_status, check_splits, compare_runs,
)

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/hepml").is_dir())
SETTINGS = Path(os.environ.get("HEPML_RESEARCH_CONFIG", REPO / "notebooks/settings.local.json"))
config = read_settings(SETTINGS)
display(pd.Series(config, name="Effective settings"))
print(f"Luminosity: {config['analysis']['lumi']:g} pb^-1 ({config['analysis']['lumi'] / 1000:g} fb^-1)")


In [ ]:
RUN_IDS = [config["run_id"]]  # Add other completed run IDs on this benchmark.
REVEAL_TEST = False
comparison_split = "test" if REVEAL_TEST else "val"
run_dirs = [Path(config["workspace"]) / "runs" / name for name in RUN_IDS]

## Results at each model's validation-selected threshold
The table reports **full prepared-data** yields and significance, normalized per sample.
Raw `*_in_split` columns remain for auditing. Old runs retain their original validation
threshold and are labeled `legacy_per_class` in `threshold_normalization`. A false
`validation_threshold_valid` means no
validation threshold satisfied the configured scan criteria; do not treat its
fallback threshold as an accepted sensitivity result. No threshold is selected on test.

In [ ]:
comparison = compare_runs(run_dirs, split=comparison_split)
display(comparison)

## Weighted ROC curves

In [ ]:
from sklearn.metrics import roc_curve
from hepml.adapters.research import evaluation_weights_for_run
roc_fig, ax = plt.subplots(figsize=(6, 5))
for directory in run_dirs:
    settings = json.loads((directory / "provenance.json").read_text())["settings"]
    predictions = pd.read_parquet(directory / "models" / f"sig{settings['mass']}" / f"preds_{comparison_split}.parquet")
    weights, _, _ = evaluation_weights_for_run(directory, predictions)
    fpr, tpr, _ = roc_curve(predictions.target, predictions.bdt_score, sample_weight=weights)
    ax.plot(fpr, tpr, label=directory.name)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set(xlabel="Weighted background efficiency", ylabel="Weighted signal efficiency", title=f"{comparison_split}: saved predictions")
ax.legend()
roc_fig.tight_layout()
plt.show()

## Save the comparison

In [ ]:
from datetime import datetime, timezone
report_dir = Path(config["workspace"]) / "reports" / ("comparison-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
report_dir.mkdir(parents=True, exist_ok=False)
comparison.to_csv(report_dir / "comparison.csv", index=False)
(report_dir / "inputs.json").write_text(json.dumps({"runs": [str(p) for p in run_dirs], "split": comparison_split}, indent=2))
roc_fig.savefig(report_dir / "roc.png", dpi=150)
print(report_dir)